**Silver → Gold notebook code (PySpark)**

In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F

# Read Silver table
silver_df = spark.table('silver_food_orders')

# Optional: enable write optimizations for Gold session if needed
spark.conf.set('spark.microsoft.delta.optimizeWrite.enabled', 'true')
# Enable V-Order only for read-heavy curated writes if your workspace/table strategy requires it
# spark.conf.set('spark.sql.parquet.vorder.default', 'true')

# Build Gold fact-style table
gold_orders = (silver_df
    .withColumn('order_date', F.to_date('order_time'))
    .withColumn('order_hour', F.hour('order_time'))
    .withColumn('day_name', F.date_format('order_time', 'E'))
    .withColumn('is_weekend', F.dayofweek('order_time').isin(1,7))
    .select(
        'order_id','order_time','delivery_time','restaurant','cuisine',
        'location','city','online_order','book_table',
        'total_amount','distance_km','rating_num','delivery_minutes',
        'order_date','order_hour','day_name','is_weekend'
    )
)

# Save as managed Delta table in Gold layer
(gold_orders.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable('gold_orders'))

# Optional: also save a Parquet file export for sharing or downstream use
(gold_orders.write
    .mode('overwrite')
    .format('parquet')
    .save('Files/gold/gold_orders_parquet'))

# Example dimension tables
dim_restaurant = (gold_orders
    .select('restaurant','cuisine')
    .dropna()
    .dropDuplicates())

dim_location = (gold_orders
    .select('location','city')
    .dropna()
    .dropDuplicates())

(dim_restaurant.write.mode('overwrite').format('delta').saveAsTable('dim_restaurant'))
(dim_location.write.mode('overwrite').format('delta').saveAsTable('dim_location'))

# Optional maintenance
# spark.sql('OPTIMIZE gold_orders')



StatementMeta(, 7c959722-9da6-434e-8423-a51b2f2f172e, 4, Finished, Available, Finished, False)